In [1]:
# Installation des dépendances
!pip install kagglehub scipy

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Import des bibliothèques
import os
import shutil
import json
import re
import scipy.io
import numpy as np
from tqdm.notebook import tqdm

In [3]:
# Téléchargement du dataset CompCars depuis Kaggle
print("📥 Téléchargement du dataset CompCars...")

import kagglehub

# Télécharger la dernière version
dataset_path = kagglehub.dataset_download("renancostaalencar/compcars")
print(f"✅ Dataset téléchargé: {dataset_path}")

# Afficher la structure
print("\n📁 Structure du dataset:")
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    if os.path.isdir(item_path):
        size = sum(os.path.getsize(os.path.join(dirpath, filename)) 
                 for dirpath, dirnames, filenames in os.walk(item_path) 
                 for filename in filenames) / (1024**3)
        print(f"📁 {item}/ ({size:.2f} GB)")
    else:
        print(f"📄 {item}")

📥 Téléchargement du dataset CompCars...
✅ Dataset téléchargé: /home/developer/.cache/kagglehub/datasets/renancostaalencar/compcars/versions/1

📁 Structure du dataset:
📁 train_test_split/ (0.01 GB)
📁 misc/ (0.00 GB)
📁 label/ (0.00 GB)
📁 image/ (12.28 GB)
📁 part/ (3.14 GB)


In [4]:
# # Extraction des noms depuis le fichier MATLAB
# print("🔍 Extraction des noms de marques et modèles...")

# mat_file_path = os.path.join(dataset_path, "misc", "make_model_name.mat")

# # Charger le fichier .mat
# mat_data = scipy.io.loadmat(mat_file_path)
# print("✅ Fichier .mat chargé")

# # Fonction pour nettoyer les noms MATLAB
# def clean_matlab_name(raw_name):
#     """Nettoie les noms extraits des fichiers MATLAB"""
#     if isinstance(raw_name, np.ndarray):
#         raw_name = str(raw_name.item())
    
#     # Nettoyage des artefacts MATLAB
#     clean_name = re.sub(r'array\(|dtype=|[Uu]\d+|__|[\'\"\(\)]', '', str(raw_name))
#     clean_name = re.sub(r'\s+', ' ', clean_name).strip()
    
#     # Supprimer les underscores multiples et les nombres seuls
#     clean_name = re.sub(r'_+', '_', clean_name)
#     clean_name = re.sub(r'\b\d+\b', '', clean_name)
#     clean_name = clean_name.strip('_')
    
#     return clean_name if clean_name else f"Unknown_{hash(raw_name)}"

# # Extraire les marques et modèles
# makes = {}
# models = {}

# for key in mat_data.keys():
#     if not key.startswith('__'):
#         data = mat_data[key]
        
#         if 'make' in key.lower() and isinstance(data, np.ndarray):
#             for i, item in enumerate(data):
#                 makes[str(i+1)] = clean_matlab_name(item)
                
#         elif 'model' in key.lower() and isinstance(data, np.ndarray):
#             for i, item in enumerate(data):
#                 models[str(i+1)] = clean_matlab_name(item)

# print(f"✅ {len(makes)} marques extraites")
# print(f"✅ {len(models)} modèles extraits")

# # Afficher quelques exemples
# print("\n🔍 Exemples de marques:")
# for make_id, make_name in list(makes.items())[:5]:
#     print(f"   {make_id} → {make_name}")

# print("\n🔍 Exemples de modèles:")
# for model_id, model_name in list(models.items())[:5]:
#     print(f"   {model_id} → {model_name}")

# # Sauvegarder les mappings
# mappings = {'makes': makes, 'models': models}

🔍 Extraction des noms de marques et modèles...
✅ Fichier .mat chargé
✅ 163 marques extraites
✅ 2004 modèles extraits

🔍 Exemples de marques:
   1 → [ABT]
   2 → [BAC]
   3 → [Conquest]
   4 → [DS]
   5 → [Dacia]

🔍 Exemples de modèles:
   1 → [Audi A3 hatchback]
   2 → [Audi A4L]
   3 → [Audi A6L]
   4 → [Audi Q3]
   5 → [Audi Q5]


In [5]:
# # Fonction de conversion avec noms propres
# def convert_compcars_to_imagefolder(source_path, output_path, mappings, max_makes=50, max_models=20, max_images=50):
#     """
#     Convertit CompCars au format ImageFolder avec noms propres
#     """
#     print(f"🔄 Conversion vers {output_path}")
    
#     # Créer les dossiers de sortie
#     train_dir = os.path.join(output_path, 'train')
#     test_dir = os.path.join(output_path, 'test')
#     os.makedirs(train_dir, exist_ok=True)
#     os.makedirs(test_dir, exist_ok=True)
    
#     # Chemin des images
#     image_base = os.path.join(source_path, 'image')
    
#     if not os.path.exists(image_base):
#         print("❌ Dossier images non trouvé")
#         return False
    
#     # Structures pour le suivi
#     class_mapping = {}
#     real_name_mapping = {}
#     all_images = []
    
#     # Parcourir les marques
#     makes = sorted([d for d in os.listdir(image_base) if os.path.isdir(os.path.join(image_base, d))])
#     print(f"📊 {len(makes)} marques trouvées")
    
#     for make_id in tqdm(makes[:max_makes], desc="Marques"):
#         make_path = os.path.join(image_base, make_id)
#         make_name = mappings['makes'].get(make_id, f"Make_{make_id}")
        
#         # Parcourir les modèles
#         models = sorted([d for d in os.listdir(make_path) if os.path.isdir(os.path.join(make_path, d))])
        
#         for model_id in models[:max_models]:
#             model_path = os.path.join(make_path, model_id)
#             model_name = mappings['models'].get(model_id, f"Model_{model_id}")
            
#             # Créer le nom de classe
#             class_name = f"{make_name}_{model_name}"
#             class_safe_name = re.sub(r'[^\w]', '_', class_name)  # Nettoyer pour système de fichiers
            
#             class_mapping[class_safe_name] = len(class_mapping)
#             real_name_mapping[class_safe_name] = {
#                 'original_name': class_name,
#                 'make_id': make_id,
#                 'make_name': make_name,
#                 'model_id': model_id,
#                 'model_name': model_name
#             }
            
#             # Créer les dossiers
#             os.makedirs(os.path.join(train_dir, class_safe_name), exist_ok=True)
#             os.makedirs(os.path.join(test_dir, class_safe_name), exist_ok=True)
            
#             # Collecter les images
#             images_found = 0
#             for root, dirs, files in os.walk(model_path):
#                 for file in files[:max_images]:
#                     if file.lower().endswith(('.jpg', '.jpeg', '.png')):
#                         img_path = os.path.join(root, file)
#                         all_images.append((img_path, class_safe_name))
#                         images_found += 1
#                 break  # Premier niveau seulement
            
#             if images_found > 0:
#                 print(f"   🖼️  {class_name}: {images_found} images")
    
#     # Séparation train/test
#     import random
#     random.seed(42)
#     random.shuffle(all_images)
    
#     split_idx = int(len(all_images) * 0.8)
#     train_images = all_images[:split_idx]
#     test_images = all_images[split_idx:]
    
#     # Copie des images
#     print("📦 Copie des images...")
    
#     def copy_images(images, destination):
#         copied = 0
#         for img_path, class_name in tqdm(images, desc="Copie"):
#             try:
#                 filename = os.path.basename(img_path)
#                 dest_path = os.path.join(destination, class_name, filename)
                
#                 if not os.path.exists(dest_path):
#                     shutil.copy2(img_path, dest_path)
                
#                 copied += 1
#             except Exception as e:
#                 continue
        
#         return copied
    
#     train_copied = copy_images(train_images, train_dir)
#     test_copied = copy_images(test_images, test_dir)
    
#     # Sauvegarde des métadonnées
#     with open(os.path.join(output_path, 'class_mapping.json'), 'w') as f:
#         json.dump(class_mapping, f, indent=2)
    
#     with open(os.path.join(output_path, 'real_names.json'), 'w') as f:
#         json.dump(real_name_mapping, f, indent=2, ensure_ascii=False)
    
#     with open(os.path.join(output_path, 'original_mappings.json'), 'w') as f:
#         json.dump(mappings, f, indent=2, ensure_ascii=False)
    
#     print(f"🎉 Conversion terminée!")
#     print(f"   ✅ Train: {train_copied} images")
#     print(f"   ✅ Test: {test_copied} images")
#     print(f"   ✅ Classes: {len(class_mapping)}")
    
#     return True

🚀 Lancement de la conversion COMPLÈTE optimisée...
🔄 Conversion COMPLÈTE de CompCars...
🔍 Analyse de la structure complète...
📊 163 marques totales


Marques:   0%|          | 0/163 [00:00<?, ?it/s]

📊 Progression: 50 modèles, 0 images
📊 Progression: 100 modèles, 0 images
📊 Progression: 150 modèles, 0 images
📊 Progression: 200 modèles, 0 images
📊 Progression: 250 modèles, 0 images
📊 Progression: 300 modèles, 0 images
📊 Progression: 350 modèles, 0 images
📊 Progression: 400 modèles, 0 images
📊 Progression: 450 modèles, 0 images
📊 Progression: 500 modèles, 0 images
📊 Progression: 550 modèles, 0 images
📊 Progression: 600 modèles, 0 images
📊 Progression: 650 modèles, 0 images
📊 Progression: 700 modèles, 0 images
📊 Progression: 750 modèles, 0 images
📊 Progression: 800 modèles, 0 images
📊 Progression: 850 modèles, 0 images
📊 Progression: 900 modèles, 0 images
📊 Progression: 950 modèles, 0 images
📊 Progression: 1000 modèles, 0 images
📊 Progression: 1050 modèles, 0 images
📊 Progression: 1100 modèles, 0 images
📊 Progression: 1150 modèles, 0 images
📊 Progression: 1200 modèles, 0 images
📊 Progression: 1250 modèles, 0 images
📊 Progression: 1300 modèles, 0 images
📊 Progression: 1350 modèles, 0 i

Copie train: 0it [00:00, ?it/s]

Copie test: 0it [00:00, ?it/s]

🎉 Conversion COMPLÈTE terminée!
   ✅ Marques: 163
   ✅ Modèles: 1716
   ✅ Classes: 1644
   ✅ Images train: 0
   ✅ Images test: 0
   ✅ Total images: 0
✅ Conversion COMPLÈTE réussie!


In [6]:
# # Lancement de la conversion COMPLÈTE
# print("🚀 Lancement de la conversion COMPLÈTE...")

# output_path = "./compcars_complete_imagefolder"

# # Nettoyer l'output précédent
# if os.path.exists(output_path):
#     shutil.rmtree(output_path)

# # Lancer la conversion COMPLÈTE (sans limitations)
# success = convert_compcars_to_imagefolder(
#     dataset_path,
#     output_path,
#     mappings,
#     max_makes=1000,    # Traiter TOUTES les marques
#     max_models=1000,   # Traiter TOUS les modèles
#     max_images=1000    # Traiter TOUTES les images
# )

# if success:
#     print("✅ Conversion COMPLÈTE réussie!")
# else:
#     print("❌ Conversion échouée")

🚀 Lancement de la conversion COMPLÈTE...


NameError: name 'convert_compcars_to_imagefolder' is not defined

In [ ]:
# # Vérification du résultat
# print("🔍 Vérification de la conversion...")

# train_dir = os.path.join(output_path, 'train')

# if os.path.exists(train_dir):
#     classes = os.listdir(train_dir)
#     print(f"📊 {len(classes)} classes créées")
    
#     print("\n📋 Exemples de classes:")
#     for cls in classes[:10]:
#         cls_path = os.path.join(train_dir, cls)
#         images = [f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
#         print(f"   {cls}: {len(images)} images")
    
#     # Charger les noms réels
#     real_names_path = os.path.join(output_path, 'real_names.json')
#     if os.path.exists(real_names_path):
#         with open(real_names_path, 'r') as f:
#             real_names = json.load(f)
#         print(f"\n🔍 Exemples de noms réels:")
#         for cls, info in list(real_names.items())[:3]:
#             print(f"   {cls} → {info['original_name']}")
# else:
#     print("❌ Dossier train non trouvé")

In [ ]:
# # Chargement avec PyTorch
# print("🧠 Chargement avec PyTorch...")

# try:
#     from torchvision import datasets, transforms
#     from torch.utils.data import DataLoader
#     import torch
    
#     # Transformations
#     transform = transforms.Compose([
#         transforms.Resize((256, 256)),
#         transforms.CenterCrop(224),
#         transforms.ToTensor(),
#         transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
#     ])
    
#     # Chargement du dataset
#     dataset = datasets.ImageFolder(root=train_dir, transform=transform)
    
#     print("✅ Dataset chargé!")
#     print(f"📊 Classes: {len(dataset.classes)}")
#     print(f"📊 Images: {len(dataset)}")
#     print(f"📋 Exemples: {dataset.classes[:5]}")
    
#     # DataLoader
#     dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
#     batch = next(iter(dataloader))
    
#     print(f"📦 Batch shape: {batch[0].shape}")
#     print(f"🏷️  Labels: {batch[1][:10]}")
    
# except Exception as e:
#     print(f"❌ Erreur: {e}")

In [ ]:
# # Sauvegarde des chemins pour usage futur
# print("💾 Sauvegarde des chemins...")

# paths_info = {
#     "dataset_path": dataset_path,
#     "output_path": output_path,
#     "train_dir": os.path.join(output_path, 'train'),
#     "test_dir": os.path.join(output_path, 'test'),
#     "mappings_path": os.path.join(output_path, 'real_names.json')
# }

# with open('dataset_paths.json', 'w') as f:
#     json.dump(paths_info, f, indent=2)

# print("✅ Chemins sauvegardés dans dataset_paths.json")
# print("🎉 Toutes les opérations sont terminées!")

In [7]:
# CELLULE COMPLÈTE POUR LA CONVERSION TOTALE
print("🚀 PRÉPARATION DE LA CONVERSION COMPLÈTE")

# Réinstaller les dépendances si nécessaire
!pip install kagglehub scipy tqdm --quiet

# Import des bibliothèques
import os
import shutil
import json
import re
import scipy.io
import numpy as np
from tqdm.notebook import tqdm

# Chemin du dataset
dataset_path = "/home/developer/.cache/kagglehub/datasets/renancostaalencar/compcars/versions/1"

# Charger les mappings existants ou les recréer
def load_or_create_mappings():
    """Charge les mappings ou les crée si nécessaire"""
    mappings_path = "./compcars_mappings.json"
    
    if os.path.exists(mappings_path):
        print("📖 Chargement des mappings existants...")
        with open(mappings_path, 'r') as f:
            return json.load(f)
    else:
        print("🔍 Création des mappings...")
        mat_file_path = os.path.join(dataset_path, "misc", "make_model_name.mat")
        mat_data = scipy.io.loadmat(mat_file_path)
        
        def clean_matlab_name(raw_name):
            if isinstance(raw_name, np.ndarray):
                raw_name = str(raw_name.item())
            clean_name = re.sub(r'array\(|dtype=|[Uu]\d+|__|[\'\"\(\)]', '', str(raw_name))
            clean_name = re.sub(r'\s+', ' ', clean_name).strip()
            clean_name = re.sub(r'_+', '_', clean_name)
            clean_name = re.sub(r'\b\d+\b', '', clean_name)
            return clean_name.strip('_') or f"Unknown_{hash(raw_name)}"
        
        makes, models = {}, {}
        for key in mat_data.keys():
            if not key.startswith('__'):
                data = mat_data[key]
                if 'make' in key.lower() and isinstance(data, np.ndarray):
                    for i, item in enumerate(data):
                        makes[str(i+1)] = clean_matlab_name(item)
                elif 'model' in key.lower() and isinstance(data, np.ndarray):
                    for i, item in enumerate(data):
                        models[str(i+1)] = clean_matlab_name(item)
        
        mappings = {'makes': makes, 'models': models}
        with open(mappings_path, 'w') as f:
            json.dump(mappings, f, indent=2)
        
        return mappings

# Charger les mappings
mappings = load_or_create_mappings()
print(f"✅ {len(mappings['makes'])} marques, {len(mappings['models'])} modèles")

# FONCTION DE CONVERSION COMPLÈTE
def convert_complete_compcars(source_path, output_path, mappings):
    """Conversion optimisée pour le dataset COMPLET CompCars"""
    print("🔄 Conversion COMPLÈTE de CompCars...")
    
    # Créer les dossiers de sortie
    train_dir = os.path.join(output_path, 'train')
    test_dir = os.path.join(output_path, 'test')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    # Chemin des images
    image_base = os.path.join(source_path, 'image')
    
    # Structures pour le suivi
    class_mapping = {}
    real_name_mapping = {}
    all_images = []
    
    # COMPTEUR pour suivre la progression
    total_makes = 0
    total_models = 0
    total_images = 0
    
    print("🔍 Analyse de la structure complète...")
    
    # Parcourir TOUTES les marques
    makes = sorted([d for d in os.listdir(image_base) if os.path.isdir(os.path.join(image_base, d))])
    total_makes = len(makes)
    print(f"📊 {total_makes} marques totales")
    
    for make_id in tqdm(makes, desc="Marques"):
        make_path = os.path.join(image_base, make_id)
        make_name = mappings['makes'].get(make_id, f"Make_{make_id}")
        
        # Parcourir TOUS les modèles
        models = sorted([d for d in os.listdir(make_path) if os.path.isdir(os.path.join(make_path, d))])
        
        for model_id in models:
            model_path = os.path.join(make_path, model_id)
            model_name = mappings['models'].get(model_id, f"Model_{model_id}")
            
            # Créer le nom de classe
            class_name = f"{make_name}_{model_name}"
            class_safe_name = re.sub(r'[^\w]', '_', class_name)
            
            class_mapping[class_safe_name] = len(class_mapping)
            real_name_mapping[class_safe_name] = {
                'original_name': class_name,
                'make_id': make_id,
                'make_name': make_name,
                'model_id': model_id,
                'model_name': model_name
            }
            
            # Créer les dossiers
            os.makedirs(os.path.join(train_dir, class_safe_name), exist_ok=True)
            os.makedirs(os.path.join(test_dir, class_safe_name), exist_ok=True)
            
            # Collecter TOUTES les images
            images_found = 0
            for root, dirs, files in os.walk(model_path):
                for file in files:
                    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        img_path = os.path.join(root, file)
                        all_images.append((img_path, class_safe_name))
                        images_found += 1
                        total_images += 1
                break  # Premier niveau seulement
            
            total_models += 1
            
            # Afficher progression tous les 50 modèles
            if total_models % 50 == 0:
                print(f"📊 Progression: {total_models} modèles, {total_images} images")
    
    print(f"✅ Analyse terminée: {total_makes} marques, {total_models} modèles, {total_images} images")
    
    # Séparation train/test
    import random
    random.seed(42)
    random.shuffle(all_images)
    
    split_idx = int(len(all_images) * 0.8)
    train_images = all_images[:split_idx]
    test_images = all_images[split_idx:]
    
    print(f"📊 Split: {len(train_images)} train, {len(test_images)} test")
    
    # Copie des images avec barre de progression
    print("📦 Copie des images...")
    
    def copy_images(images, destination, desc):
        copied = 0
        for img_path, class_name in tqdm(images, desc=desc):
            try:
                filename = os.path.basename(img_path)
                dest_path = os.path.join(destination, class_name, filename)
                
                if not os.path.exists(dest_path):
                    shutil.copy2(img_path, dest_path)
                
                copied += 1
            except Exception as e:
                continue
        
        return copied
    
    train_copied = copy_images(train_images, train_dir, "Copie train")
    test_copied = copy_images(test_images, test_dir, "Copie test")
    
    # Sauvegarde des métadonnées
    with open(os.path.join(output_path, 'class_mapping.json'), 'w') as f:
        json.dump(class_mapping, f, indent=2)
    
    with open(os.path.join(output_path, 'real_names.json'), 'w') as f:
        json.dump(real_name_mapping, f, indent=2, ensure_ascii=False)
    
    with open(os.path.join(output_path, 'stats.json'), 'w') as f:
        stats = {
            'total_makes': total_makes,
            'total_models': total_models,
            'total_images': total_images,
            'train_images': train_copied,
            'test_images': test_copied,
            'total_classes': len(class_mapping)
        }
        json.dump(stats, f, indent=2)
    
    print(f"🎉 Conversion COMPLÈTE terminée!")
    print(f"   ✅ Marques: {total_makes}")
    print(f"   ✅ Modèles: {total_models}")
    print(f"   ✅ Classes: {len(class_mapping)}")
    print(f"   ✅ Images train: {train_copied}")
    print(f"   ✅ Images test: {test_copied}")
    print(f"   ✅ Total images: {train_copied + test_copied}")
    
    return True

# LANCEMENT DE LA CONVERSION
print("🚀 Lancement de la conversion COMPLÈTE...")

output_path_complete = "./compcars_complete_imagefolder"

# Nettoyer l'output précédent
if os.path.exists(output_path_complete):
    shutil.rmtree(output_path_complete)

# Lancer la conversion
success = convert_complete_compcars(dataset_path, output_path_complete, mappings)

if success:
    print("✅ Conversion COMPLÈTE réussie!")
else:
    print("❌ Conversion échouée")

🚀 PRÉPARATION DE LA CONVERSION COMPLÈTE
🔍 Création des mappings...
✅ 163 marques, 2004 modèles
🚀 Lancement de la conversion COMPLÈTE...
🔄 Conversion COMPLÈTE de CompCars...
🔍 Analyse de la structure complète...
📊 163 marques totales


Marques:   0%|          | 0/163 [00:00<?, ?it/s]

📊 Progression: 50 modèles, 0 images
📊 Progression: 100 modèles, 0 images
📊 Progression: 150 modèles, 0 images
📊 Progression: 200 modèles, 0 images
📊 Progression: 250 modèles, 0 images
📊 Progression: 300 modèles, 0 images
📊 Progression: 350 modèles, 0 images
📊 Progression: 400 modèles, 0 images
📊 Progression: 450 modèles, 0 images
📊 Progression: 500 modèles, 0 images
📊 Progression: 550 modèles, 0 images
📊 Progression: 600 modèles, 0 images
📊 Progression: 650 modèles, 0 images
📊 Progression: 700 modèles, 0 images
📊 Progression: 750 modèles, 0 images
📊 Progression: 800 modèles, 0 images
📊 Progression: 850 modèles, 0 images
📊 Progression: 900 modèles, 0 images
📊 Progression: 950 modèles, 0 images
📊 Progression: 1000 modèles, 0 images
📊 Progression: 1050 modèles, 0 images
📊 Progression: 1100 modèles, 0 images
📊 Progression: 1150 modèles, 0 images
📊 Progression: 1200 modèles, 0 images
📊 Progression: 1250 modèles, 0 images
📊 Progression: 1300 modèles, 0 images
📊 Progression: 1350 modèles, 0 i

Copie train: 0it [00:00, ?it/s]

Copie test: 0it [00:00, ?it/s]

🎉 Conversion COMPLÈTE terminée!
   ✅ Marques: 163
   ✅ Modèles: 1716
   ✅ Classes: 1644
   ✅ Images train: 0
   ✅ Images test: 0
   ✅ Total images: 0
✅ Conversion COMPLÈTE réussie!
